### 1st-hour OSI, PIP, PEEP with TWs

In [1]:
# For V3: OSI (computed) + PIP/PEEP (existing) mean/std by Tumbling_window -> save to Sheet5
import os
import re
import pandas as pd

# === Input paths ===
csv_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/TWs_V3_1st"
excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/PARDS_Risk_V3_df.xlsx"

def norm_key(s: str) -> str:
    """Normalize keys to maximize match rate."""
    s = "" if s is None else str(s)
    s = s.strip()
    s = s.replace(".csv", "")
    s = s.replace("_Raw", "")
    s = re.sub(r"\s+", "", s)  # remove hidden whitespace/newlines
    return s

# === Read Sheet4 and prepare Sheet5 ===
df_excel = pd.read_excel(excel_path, sheet_name="Sheet4")

if "1st_rel_fname" not in df_excel.columns:
    raise ValueError("Sheet4 is missing required column: '1st_rel_fname'")

df_sheet5 = df_excel.copy()
df_sheet5["1st_rel_fname_key"] = df_sheet5["1st_rel_fname"].map(norm_key)

# === Add output columns for each metric and TW ===
for metric in ["OSI", "PIP", "PEEP"]:
    for tw in range(1, 7):
        df_sheet5[f"{metric}_mean_TW{tw}"] = pd.NA
        df_sheet5[f"{metric}_std_TW{tw}"] = pd.NA

# === Signal candidates ===
# Paw candidates (vent-dependent)
paw_candidates = [
    "CAPSULE_AVEAA_VITAL_4729",            # AVEA - Paw
    "CAPSULE_DRAGERMEDIBUS_VITAL_1415",    # CDGR - Paw
    "CAPSULE_MAQUETC_VITAL_1415",          # SVU  - Paw
]

# FiO2 candidates (vent-dependent)
fio2_candidates = [
    "CAPSULE_AVEAA_VITAL_2343",            # AVEA - FiO2
    "CAPSULE_DRAGERMEDIBUS_VITAL_635",     # CDGR - FiO2
    "CAPSULE_MAQUETC_VITAL_635",           # SVU  - FiO2
]

# SpO2
spo2_col = "PARM_SPO2_1"

# PIP/PEEP candidates (same as your code)
pip_candidates = [
    "CAPSULE_AVEAA_VITAL_2564",
    "CAPSULE_DRAGERMEDIBUS_VITAL_2564",
    "CAPSULE_MAQUETC_VITAL_1414"
]
peep_candidates = [
    "CAPSULE_AVEAA_VITAL_1189",
    "CAPSULE_DRAGERMEDIBUS_VITAL_1189",
    "CAPSULE_MAQUETC_VITAL_1189"
]

# Fast lookup from key -> indices (handles duplicates)
key_to_indices = df_sheet5.groupby("1st_rel_fname_key").indices

# ---- Debug counters (optional but useful) ----
n_files = 0
n_key_matched = 0
n_missing_tw = 0
n_missing_for_osi = 0
n_updated_any = 0

# === Loop through CSVs ===
for filename in os.listdir(csv_dir):
    if not filename.endswith(".csv"):
        continue
    n_files += 1

    file_path = os.path.join(csv_dir, filename)
    df = pd.read_csv(file_path)

    # Key from filename, matches 1st_rel_fname
    # Example: 1000780_20250420_18_1st_Raw.csv -> 1000780_20250420_18_1st
    key = norm_key(filename)

    if key not in key_to_indices:
        continue
    n_key_matched += 1

    # Must have Tumbling_window
    if "Tumbling_window" not in df.columns:
        n_missing_tw += 1
        continue

    # Normalize Tumbling_window to numeric 1..6 (handles "TW1", "1", 1.0, etc.)
    tw_raw = df["Tumbling_window"]
    if tw_raw.dtype == object:
        tw_num = tw_raw.astype(str).str.extract(r"(\d+)", expand=False)
        df["Tumbling_window"] = pd.to_numeric(tw_num, errors="coerce")
    else:
        df["Tumbling_window"] = pd.to_numeric(df["Tumbling_window"], errors="coerce")

    idx_list = key_to_indices[key]

    # ---- Find usable columns and coerce to numeric ----
    # SpO2 needed for OSI
    if spo2_col in df.columns:
        df[spo2_col] = pd.to_numeric(df[spo2_col], errors="coerce")
    else:
        n_missing_for_osi += 1
        continue

    # Choose first Paw column that exists and has some data
    paw_col = next((c for c in paw_candidates if c in df.columns and df[c].dropna().size > 0), None)
    fio2_col = next((c for c in fio2_candidates if c in df.columns and df[c].dropna().size > 0), None)

    if paw_col:
        df[paw_col] = pd.to_numeric(df[paw_col], errors="coerce")
    if fio2_col:
        df[fio2_col] = pd.to_numeric(df[fio2_col], errors="coerce")

    # PIP / PEEP
    pip_col = next((c for c in pip_candidates if c in df.columns and df[c].dropna().size > 0), None)
    peep_col = next((c for c in peep_candidates if c in df.columns and df[c].dropna().size > 0), None)
    if pip_col:
        df[pip_col] = pd.to_numeric(df[pip_col], errors="coerce")
    if peep_col:
        df[peep_col] = pd.to_numeric(df[peep_col], errors="coerce")

    updated_this_file = False

    # === Compute stats within each time window ===
    for tw in range(1, 7):
        df_tw = df[df["Tumbling_window"] == tw]
        if df_tw.empty:
            continue

        # --- OSI computed on the fly: (Paw * FiO2) / SpO2 ---
        if paw_col and fio2_col:
            denom = df_tw[spo2_col].replace(0, pd.NA)
            osi_vals = (df_tw[paw_col] * df_tw[fio2_col]) / denom
            osi_vals = osi_vals.replace([float("inf"), -float("inf")], pd.NA).dropna()

            if not osi_vals.empty:
                df_sheet5.loc[idx_list, f"OSI_mean_TW{tw}"] = round(float(osi_vals.mean()), 2)
                df_sheet5.loc[idx_list, f"OSI_std_TW{tw}"] = round(float(osi_vals.std()), 2)
                updated_this_file = True
        else:
            # Can't compute OSI if we don't have both Paw and FiO2
            pass

        # --- PIP (existing column) ---
        if pip_col:
            pip_vals = df_tw[pip_col].dropna()
            if not pip_vals.empty:
                df_sheet5.loc[idx_list, f"PIP_mean_TW{tw}"] = round(float(pip_vals.mean()), 2)
                df_sheet5.loc[idx_list, f"PIP_std_TW{tw}"] = round(float(pip_vals.std()), 2)
                updated_this_file = True

        # --- PEEP (existing column) ---
        if peep_col:
            peep_vals = df_tw[peep_col].dropna()
            if not peep_vals.empty:
                df_sheet5.loc[idx_list, f"PEEP_mean_TW{tw}"] = round(float(peep_vals.mean()), 2)
                df_sheet5.loc[idx_list, f"PEEP_std_TW{tw}"] = round(float(peep_vals.std()), 2)
                updated_this_file = True

    if updated_this_file:
        n_updated_any += 1

# === Save to Sheet5 ===
df_out = df_sheet5.drop(columns=["1st_rel_fname_key"])

with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_out.to_excel(writer, sheet_name="Sheet5", index=False)

print("✅ OSI (computed), PIP, and PEEP mean/std by time window saved to Sheet5.")
print(f"Files scanned: {n_files}")
print(f"Filename key matched to Sheet4 rows: {n_key_matched}")
print(f"Matched but missing Tumbling_window: {n_missing_tw}")
print(f"Matched but missing SpO2 or OSI inputs: {n_missing_for_osi}")
print(f"Files that updated at least one TW metric: {n_updated_any}")


✅ OSI (computed), PIP, and PEEP mean/std by time window saved to Sheet5.
Files scanned: 188
Filename key matched to Sheet4 rows: 188
Matched but missing Tumbling_window: 0
Matched but missing SpO2 or OSI inputs: 0
Files that updated at least one TW metric: 188


In [ ]:
# # For V2
# import os
# import pandas as pd
# from openpyxl import load_workbook

# # === Input paths ===
# csv_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_1st_Raw/OSI_V2_1st/TWs_V2_1st/PEEP_Settings_V2_1st/Abnormal_Deletion_V2_1st/TV_V2_1st"
# excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"

# # === Read Sheet16 and prepare Sheet17 ===
# df_excel = pd.read_excel(excel_path, sheet_name="Sheet16")
# df_excel["FileName_V2_1st"] = df_excel["FileName_V2_1st"].astype(str).str.strip()
# df_sheet17 = df_excel.copy()

# # === Add output columns for each metric and TW ===
# for metric in ["OSI", "PIP", "PEEP"]:
#     for tw in range(1, 7):
#         df_sheet17[f"{metric}_mean_TW{tw}"] = pd.NA
#         df_sheet17[f"{metric}_std_TW{tw}"] = pd.NA

# # Define column candidates
# osi_col = "OSI"
# pip_candidates = [
#     "CAPSULE_AVEAA_VITAL_2564",
#     "CAPSULE_DRAGERMEDIBUS_VITAL_2564",
#     "CAPSULE_MAQUETC_VITAL_1414"
# ]
# peep_candidates = [
#     "CAPSULE_AVEAA_VITAL_1189",
#     "CAPSULE_DRAGERMEDIBUS_VITAL_1189",
#     "CAPSULE_MAQUETC_VITAL_1189"
# ]

# # === Loop through CSVs ===
# for filename in os.listdir(csv_dir):
#     if not filename.endswith(".csv"):
#         continue

#     basename = filename.replace(".csv", "").strip()
#     file_path = os.path.join(csv_dir, filename)
#     df = pd.read_csv(file_path)

#     match = df_sheet17[df_sheet17["FileName_V2_1st"] == basename]
#     if match.empty or "Tumbling_window" not in df.columns or osi_col not in df.columns:
#         continue

#     pip_col = next((col for col in pip_candidates if col in df.columns and not df[col].dropna().empty), None)
#     peep_col = next((col for col in peep_candidates if col in df.columns and not df[col].dropna().empty), None)

#     for tw in range(1, 7):
#         df_tw = df[df["Tumbling_window"] == tw]
#         if df_tw.empty:
#             continue

#         # OSI
#         osi_values = df_tw[osi_col].dropna()
#         if not osi_values.empty:
#             df_sheet17.loc[match.index, f"OSI_mean_TW{tw}"] = round(osi_values.mean(), 2)
#             df_sheet17.loc[match.index, f"OSI_std_TW{tw}"] = round(osi_values.std(), 2)

#         # PIP
#         if pip_col:
#             pip_values = df_tw[pip_col].dropna()
#             if not pip_values.empty:
#                 df_sheet17.loc[match.index, f"PIP_mean_TW{tw}"] = round(pip_values.mean(), 2)
#                 df_sheet17.loc[match.index, f"PIP_std_TW{tw}"] = round(pip_values.std(), 2)

#         # PEEP
#         if peep_col:
#             peep_values = df_tw[peep_col].dropna()
#             if not peep_values.empty:
#                 df_sheet17.loc[match.index, f"PEEP_mean_TW{tw}"] = round(peep_values.mean(), 2)
#                 df_sheet17.loc[match.index, f"PEEP_std_TW{tw}"] = round(peep_values.std(), 2)

# # === Save to Sheet17 ===
# # book = load_workbook(excel_path)
# with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     # writer.book = book
#     df_sheet17.to_excel(writer, sheet_name="Sheet17", index=False)

# print("✅ OSI, PIP, and PEEP mean/std by time window saved to Sheet17.")


✅ OSI, PIP, and PEEP mean/std by time window saved to Sheet17.


In [ ]:
# # For V1
# import os
# import pandas as pd

# # === Input paths ===
# csv_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/VitalData/01_Vital_Raw_282_1st/OSI_285_1st/TWs_284_1st/PEEP_Settings_284_1st/Abnormal_Detection_282_1st/TV_V1_1st"
# excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1_df.xlsx"

# # === Read Sheet10 and prepare Sheet11 ===
# df_excel = pd.read_excel(excel_path, sheet_name="Sheet10")
# df_excel["FileName_Vital_1st"] = df_excel["FileName_Vital_1st"].astype(str).str.strip()
# df_sheet11 = df_excel.copy()

# # === Add output columns for each metric and time window ===
# for metric in ["OSI", "PIP", "PEEP"]:
#     for tw in range(1, 7):
#         df_sheet11[f"{metric}_mean_TW{tw}"] = pd.NA
#         df_sheet11[f"{metric}_std_TW{tw}"] = pd.NA

# # Define column names
# osi_col = "OSI"
# pip_candidates = ["AVEA - PIP", "CDGR - PIP", "SVU - PIP"]
# peep_candidates = ["AVEA - PEEP", "CDGR - PEEP", "SVU - PEEP"]

# # === Process each CSV ===
# for filename in os.listdir(csv_dir):
#     if not filename.endswith(".csv"):
#         continue

#     file_path = os.path.join(csv_dir, filename)
#     df = pd.read_csv(file_path)

#     match = df_sheet11[df_sheet11["FileName_Vital_1st"] == filename]
#     if match.empty or "Tumbling_window" not in df.columns or osi_col not in df.columns:
#         continue

#     pip_col = next((col for col in pip_candidates if col in df.columns and not df[col].dropna().empty), None)
#     peep_col = next((col for col in peep_candidates if col in df.columns and not df[col].dropna().empty), None)

#     for tw in range(1, 7):
#         df_tw = df[df["Tumbling_window"] == tw]
#         if df_tw.empty:
#             continue

#         # OSI
#         osi_values = df_tw[osi_col].dropna()
#         if not osi_values.empty:
#             df_sheet11.loc[match.index, f"OSI_mean_TW{tw}"] = round(osi_values.mean(), 2)
#             df_sheet11.loc[match.index, f"OSI_std_TW{tw}"] = round(osi_values.std(), 2)

#         # PIP
#         if pip_col:
#             pip_values = df_tw[pip_col].dropna()
#             if not pip_values.empty:
#                 df_sheet11.loc[match.index, f"PIP_mean_TW{tw}"] = round(pip_values.mean(), 2)
#                 df_sheet11.loc[match.index, f"PIP_std_TW{tw}"] = round(pip_values.std(), 2)

#         # PEEP
#         if peep_col:
#             peep_values = df_tw[peep_col].dropna()
#             if not peep_values.empty:
#                 df_sheet11.loc[match.index, f"PEEP_mean_TW{tw}"] = round(peep_values.mean(), 2)
#                 df_sheet11.loc[match.index, f"PEEP_std_TW{tw}"] = round(peep_values.std(), 2)

# # === Save to Sheet11 ===
# with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#     df_sheet11.to_excel(writer, sheet_name="Sheet11", index=False)

# print("✅ OSI, PIP, and PEEP mean/std by time window saved to Sheet11.")


✅ OSI, PIP, and PEEP mean/std by time window saved to Sheet11.
